# GPT-2 Training for Vietnamese Nguyễn Du Poetry Generation


Complete the functions/snipshot marked with ## TODO.

After completing this exercise, stundent are learnt to implement Vietnamese text normalization and syllable counting for preprocessing poetry data, create structured datasets by pairing lines into lục bát couplets (6-8 syllable meter) with special control tokens, initialize and configure GPT-2 models with custom special tokens and tokenizers for Vietnamese, tokenize datasets and set up training pipelines using the Hugging Face Transformers Trainer API with appropriate hyperparameters, train causal language models on Vietnamese poetry data while managing memory constraints, and evaluate generated poems by validating whether they follow the traditional lục bát poetic structure through programmatic checking.

In [1]:
# !pip install datasets
# !pip install transformers[torch]

In [2]:

# Import Required Libraries
import os
import re
import random
from typing import List, Tuple, Dict

import torch
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    DataCollatorForLanguageModeling,
    Trainer,
    TrainingArguments,
    set_seed,
)

print("Libraries imported successfully!")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device: {torch.cuda.get_device_name(0)}")

Libraries imported successfully!
PyTorch version: 2.4.1+cu124
CUDA available: True
CUDA device: NVIDIA GeForce RTX 3090


In [3]:
# Configuration and Constants
MODEL_NAME = "gpt2"
DATA_PATH = "dataset/truyenkieu.txt" 
OUT_DIR = "./gpt2_kieu_lucbat"

# Training hyperparameters (optimized for low memory usage)
SEED = 42
MAX_LEN = 128
BATCH_SIZE = 1
GRADIENT_ACCUMULATION_STEPS = 4
EPOCHS = 10  # Start with 1 epoch for testing
LR = 5e-5
WARMUP_STEPS = 100
WEIGHT_DECAY = 0.01
VAL_RATIO = 0.05

# Set random seed for reproducibility
set_seed(SEED)

print("Configuration set:")
print(f"Model: {MODEL_NAME}")
print(f"Data path: {DATA_PATH}")
print(f"Output directory: {OUT_DIR}")
print(f"Training epochs: {EPOCHS}")
print(f"Batch size: {BATCH_SIZE}")
print(f"Device: {'CUDA' if torch.cuda.is_available() else 'CPU'}")

Configuration set:
Model: gpt2
Data path: dataset/truyenkieu.txt
Output directory: ./gpt2_kieu_lucbat
Training epochs: 10
Batch size: 1
Device: CUDA


In [ ]:
# Text Preprocessing Functions for Vietnamese

# Regular expression to keep Vietnamese letters, spaces, hyphens, and apostrophes
_punct_re = re.compile(r"[^\wÀ-ỹ\s'-]+", flags=re.UNICODE)

def normalize_line(line: str) -> str:
    """Normalize Vietnamese text line for syllable counting."""
    ## TODO: Complete normalization steps
    return line

def count_syllables_approx_vn(line: str) -> int:    
    """
    Approximate Vietnamese syllable count by counting whitespace-separated words.
    This is a simple but effective method for Vietnamese.
    """
    line = normalize_line(line)
    if not line:
        return 0
    return len(line.split())

def looks_like_lucbat_pair(l1: str, l2: str, tol: int = 0) -> bool:
    """
    Check if two lines form a lục bát couplet (6-8 syllables).
    Allows tolerance for slight variations after normalization.
    """
    s1 = count_syllables_approx_vn(l1)
    s2 = count_syllables_approx_vn(l2)
    return (6 - tol <= s1 <= 6 + tol) and (8 - tol <= s2 <= 8 + tol)

# Test the functions
test_line1 = "Trăm năm trong cõi người ta"  
test_line2 = "Chữ tài chữ mệnh khéo là ghét nhau"  

print("Testing Vietnamese text processing:")
print(f"Line 1: '{test_line1}'")
print(f"Syllables: {count_syllables_approx_vn(test_line1)}")
print(f"Line 2: '{test_line2}'")
print(f"Syllables: {count_syllables_approx_vn(test_line2)}")
print(f"Is Lục Bát pair: {looks_like_lucbat_pair(test_line1, test_line2)}")

Testing Vietnamese text processing:
Line 1: 'Trăm năm trong cõi người ta'
Syllables: 6
Line 2: 'Chữ tài chữ mệnh khéo là ghét nhau'
Syllables: 8
Is Lục Bát pair: True


In [ ]:
# Dataset Preparation Functions

def load_lines(path: str) -> List[str]:
    """Load and preprocess text lines from file."""
    assert os.path.exists(path), f"Missing file: {path}"
    
    with open(path, "r", encoding="utf-8") as f:
        lines = [ln.strip() for ln in f]
    
    # Remove empty lines and normalize
    lines = [ln for ln in lines if ln.strip()]
    lines = [normalize_line(ln) for ln in lines]
    lines = [ln for ln in lines if ln]
    
    return lines

def make_couplet_samples(lines: List[str], strict: bool = True) -> List[str]:
    """
    Convert line list into training samples with special control tokens.
    Each sample is a 2-line couplet wrapped with <KIEU><LB>...</LB> tokens.
    """
    samples = []
    i = 0
    
    ## TODO
    
    return samples

def prepare_dataset(data_path: str) -> Dataset:
    """Prepare training dataset from text file."""
    print(f"Loading text from: {data_path}")
    
    # Load and preprocess lines
    raw_lines = load_lines(data_path)
    print(f"Loaded {len(raw_lines)} lines")
    print(f"First 3 lines: {raw_lines[:3]}")
    
    # Create couplet samples
    samples = make_couplet_samples(raw_lines, strict=True)
    print(f"Created {len(samples)} couplet samples")
    print(f"First training sample:\n{samples[0]}")
    
    # Shuffle and split
    random.Random(SEED).shuffle(samples)
    ds = Dataset.from_dict({"text": samples})
    ds = ds.train_test_split(test_size=VAL_RATIO, seed=SEED)
    
    return ds

# Load and prepare the dataset
dataset = prepare_dataset(DATA_PATH)
print(f"\nDataset structure: {dataset}")

Loading text from: dataset/truyenkieu.txt
Loaded 3257 lines
First 3 lines: ['Trăm năm trong cõi người ta', 'Chữ tài chữ mệnh khéo là ghét nhau', 'Trải qua một cuộc bể dâu']
Created 1628 couplet samples
First training sample:
<KIEU><LB>
Trăm năm trong cõi người ta
Chữ tài chữ mệnh khéo là ghét nhau
</LB>

Dataset structure: DatasetDict({
    train: Dataset({
        features: ['text'],
        num_rows: 1546
    })
    test: Dataset({
        features: ['text'],
        num_rows: 82
    })
})


In [ ]:
# Model and Tokenizer Setup

def setup_model_and_tokenizer():
    """Initialize tokenizer and model with special tokens."""
    print(f"Loading model: {MODEL_NAME}")
    
    ## TODO
    # Load tokenizer
    
    # Add special tokens for style control
    
    # Set pad token (GPT-2 doesn't have one by default)
    
    # Determine device and model precision
    device = "cuda" if torch.cuda.is_available() else "cpu"
    
    # Load model
    
    
    print(f"Device: {device}")
    
    return model, tokenizer, device

# Initialize model and tokenizer
model, tokenizer, device = setup_model_and_tokenizer()

Loading model: gpt2
Using CPU for training (to avoid memory issues)


The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`


Added 3 special tokens
Tokenizer vocab size: 50260
Device: cuda


In [ ]:
# Training Pipeline

def train_model(model, tokenizer, dataset):
    """Train the model on the dataset."""
    
    def tokenize_batch(batch: Dict[str, List[str]]) -> Dict[str, List[List[int]]]:
        return tokenizer(
            ## TODO: Fill in parameters
        )
    
    ## TODO: Tokenize dataset
    
    # Training arguments
    training_args = TrainingArguments(
        ## TODO: Fill in training arguments
    )
    
    # Create trainer
    trainer = Trainer(
        ## TODO: Fill in trainer parameters
    )
    
    print("Starting training...")
    
    # Train the model
    trainer.train()
    
    # Save model and tokenizer (if you want to)
    trainer.save_model(OUT_DIR)
    tokenizer.save_pretrained(OUT_DIR)
    print(f"Model saved to: {OUT_DIR}")
    
    return trainer

# Start training
trainer = train_model(model, tokenizer, dataset)

Tokenizing dataset...


Map:   0%|          | 0/1546 [00:00<?, ? examples/s]

Map:   0%|          | 0/82 [00:00<?, ? examples/s]

/tmp/ipykernel_4221/897302363.py:42: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 50256}.


Starting training...
Total training samples: 1546
Total validation samples: 82
Steps per epoch: 386


`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Step,Training Loss,Validation Loss
100,2.811200,2.648785
200,2.238200,2.157962
300,2.104700,2.047413
400,2.009400,1.927310
500,1.934400,1.874973
600,1.877000,1.833406
700,1.832500,1.796019
800,1.838200,1.769880
900,1.781900,1.763293
1000,1.778300,1.745643


Model saved to: ./gpt2_kieu_lucbat


In [ ]:
## TODO: Evaluate your model by: Generating sample poems using the trained model. You can use defined functions to check if the generated poems follow the lục bát structure.